In [13]:
grid = 1000

# 网格

In [17]:
import geopandas as gpd
from shapely.geometry import box

def export_bounding_rectangle(input_shapefile, output_shapefile):
    # 读取原始数据
    gdf = gpd.read_file(input_shapefile)

    # 坐标系转换为米单位（可选）
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)

    # 获取总边界范围
    minx, miny, maxx, maxy = gdf.total_bounds

    # 构造 rectangle 多边形
    rectangle = box(minx, miny, maxx, maxy)

    # 保存为新的 shapefile
    out_gdf = gpd.GeoDataFrame(geometry=[rectangle], crs=gdf.crs)
    out_gdf.to_file(output_shapefile)

# 示例调用
export_bounding_rectangle(
    input_shapefile=r"D:\shanghai workshop\building+park+waterbody\36上海\上海市\上海市_Polygon.shp",
    output_shapefile=r"D:\shanghai workshop\vector\shanghai_rectangle_boundary.shp"
)


In [18]:
import geopandas as gpd
from shapely.geometry import box
import numpy as np

def generate_grid_from_rectangle(input_shapefile, output_shapefile, grid_size):
    # 读取矩形边界
    rect_gdf = gpd.read_file(input_shapefile)

    # 投影为米坐标（确保单位是米）
    if not rect_gdf.crs.is_projected:
        rect_gdf = rect_gdf.to_crs(epsg=32651)

    # 获取矩形的边界范围
    minx, miny, maxx, maxy = rect_gdf.total_bounds

    # 生成格网列表
    grid_cells = []
    id_counter = 0
    for x0 in np.arange(minx, maxx, grid_size):
        for y0 in np.arange(miny, maxy, grid_size):
            x1 = x0 + grid_size
            y1 = y0 + grid_size
            cell = box(x0, y0, x1, y1)
            grid_cells.append({"geometry": cell, "ID": id_counter})
            id_counter += 1

    # 创建 GeoDataFrame
    grid_gdf = gpd.GeoDataFrame(grid_cells, crs=rect_gdf.crs)

    # 可选：裁剪掉超出 rectangle 的部分（严格限制在边界内）
    grid_gdf = gpd.overlay(grid_gdf, rect_gdf, how="intersection")

    # 保存为 shapefile
    grid_gdf.to_file(output_shapefile)

# 示例调用
generate_grid_from_rectangle(
    input_shapefile=r"D:\shanghai workshop\vector\shanghai_rectangle_boundary.shp",
    grid_size=grid,
    output_shapefile=rf"D:\shanghai workshop\vector\shanghai_grid_{grid}m.shp"
      # 单位：米
)

# 上海市切分

In [23]:
import geopandas as gpd

grid_size = grid  # 这里定义你的网格尺寸

shanghai_polygon = gpd.read_file(r"D:\shanghai workshop\building+park+waterbody\36上海\上海市\上海市_Polygon.shp")
grid = gpd.read_file(rf"D:\shanghai workshop\vector\shanghai_grid_1000m.shp")

if shanghai_polygon.crs != grid.crs:
    grid = grid.to_crs(shanghai_polygon.crs)

grid_intersect = grid[grid.geometry.intersects(shanghai_polygon.unary_union)]

output_path = rf"D:\shanghai workshop\vector\shanghai_grid_1000m_clipped.shp"
grid_intersect.to_file(output_path)

print(f"✅ 已保存筛选后的网格文件，路径：{output_path}")
print(f"筛选后保留格子数：{len(grid_intersect)}")


C:\Users\owner\AppData\Local\Temp\ipykernel_37360\2112031888.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  grid_intersect = grid[grid.geometry.intersects(shanghai_polygon.unary_union)]


✅ 已保存筛选后的网格文件，路径：D:\shanghai workshop\vector\shanghai_grid_1000m_clipped.shp
筛选后保留格子数：8500


# 01 park

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_park_by_grid(parks_shp, grid_shp, output_folder, resolution=10):
    # 读取数据
    parks = gpd.read_file(parks_shp)
    grid = gpd.read_file(grid_shp)

    # 坐标系统一（建议米制）
    if not parks.crs.is_projected:
        parks = parks.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(parks.crs)

    # 清理空 geometry
    parks = parks[~parks.geometry.is_empty & parks.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_folder, exist_ok=True)

    for idx, row in grid.iterrows():
        grid_geom = row.geometry

        # 提取该格网内的公园
        clipped = gpd.overlay(parks, gpd.GeoDataFrame(geometry=[grid_geom], crs=parks.crs), how="intersection")

        if clipped.empty:
            continue  # 如果该区域没有公园，跳过

        # 计算 raster 尺寸
        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        # 准备 raster 数据
        shapes = [(geom, 1) for geom in clipped.geometry]
        raster = rasterize(
            shapes,
            out_shape=(height, width),
            transform=transform,
            fill=0,
            dtype="uint8"
        )

        # 输出路径（按格网 ID 或索引命名）
        out_path = os.path.join(output_folder, f"park_tile_{idx}.tif")

        # 写入 raster
        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=parks.crs,
            transform=transform
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ Saved: {out_path}")

# 示例调用
rasterize_park_by_grid(
    parks_shp=r"D:\shanghai workshop\building+park+waterbody\landuse-公园用地\landuse_park.shp",
    grid_shp=rf"D:\shanghai workshop\vector\shanghai_grid_1000m_clipped.shp",
    output_folder=r"D:\shanghai workshop\raster\park_tiles",
    resolution=10  # 10米分辨率
)

✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_24.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_30.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_31.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_39.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_45.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_46.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_85.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_86.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_102.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_114.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_115.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_197.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_232.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_233.tif
✅ Saved: D:\shanghai workshop\raster\park_tiles\park_tile_234.tif
✅ Saved: D:\shangh

# 02 water

In [10]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import os

def rasterize_fclass_by_grid(shapefile_path, grid_path, fclass_mapping, output_dir, resolution):
    # 读取水体数据
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统一为 UTM 51N
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 保留指定类别的记录，并转换为数值
    gdf = gdf[gdf['fclass'].isin(fclass_mapping.keys())]
    gdf['class_val'] = gdf['fclass'].map(fclass_mapping)

    # 清理空 geometry
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_dir, exist_ok=True)

    for idx, cell in grid.iterrows():
        grid_geom = cell.geometry

        # 裁剪出当前格子内的水体
        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")

        if clipped.empty:
            continue  # 如果没有水体，跳过

        # 计算输出 raster 尺寸
        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        # 构造 shapes
        shapes = [(geom, val) for geom, val in zip(clipped.geometry, clipped['class_val'])]

        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=255,  # Nodata 为 255
            dtype='uint8'
        )

        out_path = os.path.join(output_dir, f"water_tile_{idx}.tif")

        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="uint8",
            crs=gdf.crs,
            transform=transform,
            nodata=255
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 示例调用
fclass_mapping = {
    'dock': 0,
    'reservoir': 1,
    'riverbank': 2,
    'water': 3,
    'wetland': 4
}

rasterize_fclass_by_grid(
    shapefile_path=r"D:\BaiduNetdiskDownload\3-shanghai-latest-free.shp\gis_osm_water_a_free_1.shp",
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_1000m_clipped.shp",
    fclass_mapping=fclass_mapping,
    output_dir=r"D:\shanghai workshop\raster\water_tiles",
    resolution=10
)


✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_7.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_8.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_9.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_23.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_24.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_25.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_30.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_33.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_34.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_35.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_36.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_37.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_50.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_51.tif
✅ 输出: D:\shanghai workshop\raster\water_tiles\water_tile_52.tif
✅ 输出: D:\shanghai workshop\raster\water_til

# 03 building

In [11]:
import geopandas as gpd

gdf = gpd.read_file(r"C:\Users\owner\Desktop\上海\�Ϻ�\Shanghai_Buildings_DWG-Polygon.shp")
print(gdf.crs)

PROJCS["UTM_Modified",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",121.4348038],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [11]:
import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from rasterio.transform import from_origin
import numpy as np
import os

def rasterize_real_value_by_grid(shapefile_path, field_name, grid_path, output_dir, resolution=10):
    # 读取数据
    gdf = gpd.read_file(shapefile_path)
    grid = gpd.read_file(grid_path)

    # 坐标系统一
    if not gdf.crs.is_projected:
        gdf = gdf.to_crs(epsg=32651)
    if not grid.crs.is_projected:
        grid = grid.to_crs(epsg=32651)
    if gdf.crs != grid.crs:
        grid = grid.to_crs(gdf.crs)

    # 只保留有 Elevation 值的记录
    gdf = gdf.dropna(subset=[field_name])
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()]
    grid = grid[~grid.geometry.is_empty & grid.geometry.notnull()]

    os.makedirs(output_dir, exist_ok=True)

    for idx, cell in grid.iterrows():
        grid_geom = cell.geometry

        # 裁剪
        clipped = gpd.overlay(gdf, gpd.GeoDataFrame(geometry=[grid_geom], crs=gdf.crs), how="intersection")
        if clipped.empty:
            continue

        # 边界与栅格大小
        minx, miny, maxx, maxy = grid_geom.bounds
        width = int((maxx - minx) / resolution)
        height = int((maxy - miny) / resolution)
        transform = from_origin(minx, maxy, resolution, resolution)

        # 构造 shapes
        shapes = [
            (geom, float(val)) for geom, val in zip(clipped.geometry, clipped[field_name])
        ]

        # 栅格化为 float32，背景设为 NaN
        raster = rasterize(
            shapes=shapes,
            out_shape=(height, width),
            transform=transform,
            fill=np.nan,
            dtype="float32"
        )

        out_path = os.path.join(output_dir, f"building_tile_{idx}.tif")

        # 保存
        with rasterio.open(
            out_path,
            "w",
            driver="GTiff",
            height=height,
            width=width,
            count=1,
            dtype="float32",
            crs=gdf.crs,
            transform=transform,
            nodata=np.nan
        ) as dst:
            dst.write(raster, 1)

        print(f"✅ 输出: {out_path}")

# 示例调用
rasterize_real_value_by_grid(
    shapefile_path=r"C:\Users\owner\Desktop\上海\�Ϻ�\Shanghai_Buildings_DWG-Polygon.shp",
    field_name="Elevation",
    grid_path=r"D:\shanghai workshop\vector\shanghai_grid_1000m_clipped.shp",
    output_dir=r"D:\shanghai workshop\raster\building_tiles",
    resolution=5
)


✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_34.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_35.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_61.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_62.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_85.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_88.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_89.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_90.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_112.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_113.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_114.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_115.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_116.tif
✅ 输出: D:\shanghai workshop\raster\building_tiles\building_tile_117.tif
✅ 输出: D:\shang

# 04 road